# Experiment: Phase-2 Impact Case Analysis

This notebook accompanies a small impact-analysis case study for the formal result directory:

- formal root: `results\qtraffic_phase2_gate_bypass_fix_formal`
- output dir: `results\qtraffic_phase2_gate_bypass_fix_formal\impact_case_h36_seed0`
- primary model: `H_time_query_causal`
- baseline model: `G_time_query_static`

Selection rule:

1. Rank all test windows by recent query intensity over the last `4` history steps.
2. Keep the top `64` windows.
3. Choose the representative window by the `median_high` policy inside that pool.

Representative sample:

- sample index: `73`
- query score: `1.6100`
- target start: `2017-05-20 16:15`
- target end: `2017-05-21 01:00`

Precomputed highlights:

- `query_shock` / `dynamic`: mean_signed_delta=0.0003, mean_abs_delta=0.0016, treated_abs_delta=0.0688
- `query_shock` / `static`: mean_signed_delta=0.0000, mean_abs_delta=0.0002, treated_abs_delta=0.0096
- `speed_reduction` / `dynamic`: mean_signed_delta=-0.0242, mean_abs_delta=0.0503, treated_abs_delta=1.8952
- `speed_reduction` / `static`: mean_signed_delta=-0.0339, mean_abs_delta=0.0575, treated_abs_delta=2.2709

Note:

- This is a *counterfactual-style impact analysis under a frozen trained model*.
- The past-only dynamic graph alignment is preserved, but the dynamic graph itself is not re-estimated after intervention.


In [ ]:
# ===== 0. Notebook environment =====
from pathlib import Path
import os
import sys

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "code").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)

CODE_DIR = PROJECT_ROOT / "code"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

print("PROJECT_ROOT =", PROJECT_ROOT)
print("NOTEBOOK_DIR =", NOTEBOOK_DIR)
print("CODE_DIR =", CODE_DIR)


In [ ]:
# ===== 1. Config =====
from qtraffic_phase2_impact_case import ImpactCaseConfig

impact_cfg = ImpactCaseConfig(
    formal_result_dir='./results/qtraffic_phase2_gate_bypass_fix_formal',
    output_subdir='impact_case_h36_seed0',
    qtraffic_dir='./data/qtraffic',
    seed=0,
    horizon=36,
    primary_mode='H_time_query_causal',
    baseline_mode='G_time_query_static',
    top_pool=64,
    representative_policy='median_high',
    top_k_nodes=8,
    history_tail=4,
    query_sigma=1.0,
    speed_drop_ratio=0.2,
    max_hop=3,
)
impact_cfg


In [ ]:
# ===== 2. Rerun analysis if needed =====
from qtraffic_phase2_impact_case import run_impact_case_analysis

artifacts = run_impact_case_analysis(impact_cfg, force=False)
artifacts


In [ ]:
# ===== 3. Load exported tables =====
import pandas as pd
from pathlib import Path

output_dir = Path(artifacts["output_dir"])
summary_df = pd.read_csv(output_dir / "impact_case_summary.csv")
horizon_df = pd.read_csv(output_dir / "impact_case_horizon_effects.csv")
hop_df = pd.read_csv(output_dir / "impact_case_hop_effects.csv")
selection_df = pd.read_csv(output_dir / "impact_case_selection.csv")
summary_df, selection_df.head(10)


In [ ]:
# ===== 4. Aggregate effects =====
horizon_df.head(), hop_df


In [ ]:
# ===== 5. Case tables =====
query_nodes = pd.read_csv(output_dir / "impact_case_case_nodes_query_shock.csv")
speed_nodes = pd.read_csv(output_dir / "impact_case_case_nodes_speed_reduction.csv")
query_ts = pd.read_csv(output_dir / "impact_case_case_timeseries_query_shock.csv")
speed_ts = pd.read_csv(output_dir / "impact_case_case_timeseries_speed_reduction.csv")

query_nodes.head(20), speed_nodes.head(20)


## Figures

Aggregate effects:

![aggregate_effects](impact_case_aggregate_effects.png)

Representative-case treated-node trajectories:

![case_trajectories](impact_case_case_trajectories.png)
